# 🧠 Actividad: Embeddings, Tokens y Similitud Semántica con Ollama

---

## ¿Qué vas a aprender en esta actividad?

En esta actividad explorarás tres conceptos fundamentales para trabajar con modelos de lenguaje:

| Concepto | ¿Qué es? |
|---|---|
| **Token** | La unidad mínima de texto que procesa un modelo |
| **Embedding** | Una representación numérica (vector) del significado de un texto |
| **Similitud coseno** | Una métrica para medir qué tan parecidos son dos textos semánticamente |

## ¿Qué herramientas usarás?

- **[Ollama](https://ollama.com)** — servidor local de modelos de IA (corre directamente en Colab)
- **[Gradio](https://gradio.app)** — framework para construir interfaces web desde Python
- **[Ngrok](https://ngrok.com)** — túnel para exponer tu interfaz al mundo exterior

---

> ⚠️ **Antes de comenzar:** Asegúrate de tener tu `authtoken` de Ngrok a mano.  
> Si no te has registrado aún, ve a [ngrok.com](https://ngrok.com), crea una cuenta gratuita y copia tu token desde el dashboard.

---
## ⚙️ Celda 1 — Instalación de dependencias

En esta celda instalamos todo lo necesario:

- **Ollama**: motor que corre los modelos de embeddings localmente en Colab
- **Gradio**: construye la interfaz visual interactiva
- **httpx**: cliente HTTP moderno para comunicarnos con Ollama
- **pyngrok**: permite crear un túnel público desde Colab hacia internet

⏳ Esta celda puede tardar ~1-2 minutos la primera vez.

In [1]:
# Instalamos Ollama (el servidor que corre los modelos de embeddings)
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

# Instalamos las librerías de Python que usaremos
!pip install -q gradio httpx pyngrok

print("✅ Instalación completada")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,871 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama 

---
## 🚀 Celda 2 — Arrancar Ollama y descargar modelos

Ollama funciona como un **servidor local**: corre en segundo plano y responde a peticiones HTTP.  
En esta celda lo iniciamos y descargamos los modelos de embeddings que usaremos.

### Modelos disponibles

| Modelo | Dimensiones | Velocidad | Ideal para |
|---|---|---|---|
| `nomic-embed-text` | 768 | ⚡⚡⚡ | Uso general, textos en inglés y español |
| `mxbai-embed-large` | 1024 | ⚡⚡ | Mayor precisión semántica |
| `all-minilm` | 384 | ⚡⚡⚡⚡ | Muy liviano, ideal para pruebas rápidas |

> 💡 **¿Qué son las dimensiones?** Es el tamaño del vector que produce el modelo.  
> Más dimensiones = más información capturada, pero más memoria y tiempo de procesamiento.

In [2]:
import subprocess
import time
import httpx

# ── 1. Arrancar el servidor Ollama en segundo plano ──────────────────────────
print("🟡 Iniciando servidor Ollama...")
proceso_ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Esperamos hasta que el servidor responda (máximo 30 segundos)
for intento in range(30):
    try:
        r = httpx.get("http://localhost:11434", timeout=2)
        if r.status_code == 200:
            print("✅ Servidor Ollama activo")
            break
    except Exception:
        time.sleep(1)
else:
    print("❌ No se pudo conectar al servidor Ollama. Reinicia el runtime e intenta de nuevo.")

# ── 2. Descargar los modelos de embeddings ───────────────────────────────────
MODELOS = ["nomic-embed-text", "mxbai-embed-large", "all-minilm"]

for modelo in MODELOS:
    print(f"📥 Descargando modelo: {modelo} ...")
    resultado = subprocess.run(
        ["ollama", "pull", modelo],
        capture_output=True,
        text=True
    )
    if resultado.returncode == 0:
        print(f"   ✅ {modelo} listo")
    else:
        print(f"   ⚠️  Error al descargar {modelo}: {resultado.stderr[:100]}")

print("\n🎉 Todos los modelos están listos para usar")

🟡 Iniciando servidor Ollama...
✅ Servidor Ollama activo
📥 Descargando modelo: nomic-embed-text ...
   ✅ nomic-embed-text listo
📥 Descargando modelo: mxbai-embed-large ...
   ✅ mxbai-embed-large listo
📥 Descargando modelo: all-minilm ...
   ✅ all-minilm listo

🎉 Todos los modelos están listos para usar


---
## 📖 Concepto 1: ¿Qué es un Token?

Un **token** es la unidad mínima de texto que un modelo de lenguaje procesa.  
No es exactamente igual a una palabra: los modelos dividen el texto de una manera particular llamada **tokenización**.

### Ejemplos de tokenización

```
Texto:   "Hola mundo"
Tokens:  ["Hola", " mundo"]  →  2 tokens

Texto:   "embeddings"
Tokens:  ["embed", "dings"]  →  2 tokens (palabra poco común, se divide)

Texto:   "el gato come pescado"
Tokens:  ["el", " gato", " come", " pes", "cado"]  →  5 tokens
```

### ¿Por qué importa contar tokens?

1. **Límites de contexto**: cada modelo tiene un máximo de tokens que puede procesar
2. **Costo de API**: los servicios de IA cobran por token procesado
3. **Rendimiento**: más tokens = más tiempo de procesamiento

> 🔍 **Regla práctica**: en inglés, 1 token ≈ 0.75 palabras. En español puede ser ligeramente mayor  
> porque las palabras tienden a ser más largas.

---
## 📖 Concepto 2: ¿Qué es la Similitud Coseno?

Cuando un modelo genera un embedding, convierte tu texto en un **vector de números** en un espacio de muchas dimensiones (ej: 768 números).  
La **similitud coseno** mide el ángulo entre dos de esos vectores.

### La fórmula

$$\cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \cdot \|\vec{B}\|}$$

Donde:
- $\vec{A} \cdot \vec{B}$ es el **producto punto** entre los dos vectores
- $\|\vec{A}\|$ y $\|\vec{B}\|$ son las **normas** (magnitudes) de cada vector

### Interpretación del resultado

| Score | Significado | Ejemplo |
|---|---|---|
| **~1.0** | Textos casi idénticos semánticamente | *"perro"* vs *"can"* |
| **~0.7** | Alta similitud temática | *"gato"* vs *"felino"* |
| **~0.5** | Relación temática moderada | *"gato"* vs *"animal"* |
| **~0.2** | Poca relación | *"gato"* vs *"automóvil"* |
| **~0.0** | Sin relación semántica | *"gato"* vs *"quantum"* |
| **< 0** | Conceptos opuestos | *"bueno"* vs *"malo"* |

### ¿Por qué coseno y no distancia euclidiana?

La distancia euclidiana mide **cuán lejos** están dos puntos. El coseno mide **en qué dirección apuntan**.  
Para el significado semántico, la **dirección** importa más que la magnitud.  
Un texto corto y uno largo sobre el mismo tema pueden tener vectores de diferente magnitud pero apuntar en la misma dirección.

```
          ↗ "El gato duerme"           ← misma dirección semántica
         /
    ────/──────────────────────→
       /
      ↗ "El felino descansa"          θ (ángulo pequeño) → coseno cercano a 1
```

---
## 🔧 Celda 3 — Funciones principales

Aquí definimos las tres funciones que alimentan la interfaz:

- `contar_tokens()` — pide a Ollama que procese el texto y devuelve el conteo de tokens
- `obtener_embedding()` — devuelve el vector numérico completo para un texto
- `similitud_coseno()` — aplica la fórmula matemática sobre dos vectores

In [3]:
import httpx
import math

OLLAMA_URL = "http://localhost:11434"

# ── Función 1: Contar tokens ─────────────────────────────────────────────────
def contar_tokens(texto: str, modelo: str) -> dict:
    """
    Envía el texto al endpoint /api/embed de Ollama.
    Ollama procesa el texto y devuelve metadata incluyendo
    'prompt_eval_count': el número de tokens que usó.
    """
    if not texto.strip():
        return {"error": "El texto está vacío"}

    try:
        respuesta = httpx.post(
            f"{OLLAMA_URL}/api/embed",
            json={"model": modelo, "input": texto},
            timeout=30
        )
        datos = respuesta.json()

        # Ollama devuelve el conteo de tokens en 'prompt_eval_count'
        tokens = datos.get("prompt_eval_count", None)

        if tokens is None:
            return {"error": "El modelo no devolvió conteo de tokens. Prueba con otro modelo."}

        palabras = len(texto.split())
        ratio = round(tokens / palabras, 2) if palabras > 0 else 0

        return {
            "tokens": tokens,
            "palabras": palabras,
            "ratio_tokens_por_palabra": ratio
        }

    except Exception as e:
        return {"error": str(e)}


# ── Función 2: Obtener embedding ─────────────────────────────────────────────
def obtener_embedding(texto: str, modelo: str) -> list:
    """
    Solicita a Ollama el embedding del texto.
    Devuelve una lista de números flotantes (el vector).
    """
    if not texto.strip():
        raise ValueError("El texto está vacío")

    respuesta = httpx.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": modelo, "input": texto},
        timeout=30
    )
    datos = respuesta.json()

    # El campo 'embeddings' es una lista de listas (una por input)
    embeddings = datos.get("embeddings", [])
    if not embeddings:
        raise ValueError("El modelo no devolvió embeddings")

    return embeddings[0]  # Devolvemos el vector del primer (y único) input


# ── Función 3: Similitud coseno ──────────────────────────────────────────────
def similitud_coseno(vec_a: list, vec_b: list) -> float:
    """
    Implementación manual de la similitud coseno.
    cos(θ) = (A · B) / (|A| * |B|)

    Retorna un valor entre -1 y 1.
    """
    # Producto punto: suma de multiplicaciones elemento a elemento
    producto_punto = sum(a * b for a, b in zip(vec_a, vec_b))

    # Norma (magnitud) de cada vector: raíz de la suma de cuadrados
    norma_a = math.sqrt(sum(a ** 2 for a in vec_a))
    norma_b = math.sqrt(sum(b ** 2 for b in vec_b))

    # Evitamos división por cero
    if norma_a == 0 or norma_b == 0:
        return 0.0

    return producto_punto / (norma_a * norma_b)


def interpretar_similitud(score: float) -> str:
    """Convierte el score numérico en una descripción comprensible."""
    if score >= 0.90:
        return "🟢 Textos casi idénticos semánticamente"
    elif score >= 0.75:
        return "🟢 Alta similitud semántica"
    elif score >= 0.55:
        return "🟡 Similitud moderada — temas relacionados"
    elif score >= 0.30:
        return "🟠 Baja similitud — poca relación temática"
    else:
        return "🔴 Sin similitud — textos semánticamente distintos"


print("✅ Funciones definidas correctamente")

✅ Funciones definidas correctamente


---
## 🎨 Celda 4 — Interfaz Gradio

Construimos la interfaz con tres pestañas (tabs), una por cada concepto:  

1. **Tab Tokens** — Pega texto y ve cuántos tokens genera cada modelo
2. **Tab Embedding** — Visualiza el vector numérico que produce el modelo
3. **Tab Similitud** — Compara dos textos y mide su cercanía semántica

Ejecuta esta celda y haz clic en el enlace que aparece para abrir la interfaz.

In [4]:
import gradio as gr

MODELOS_DISPONIBLES = ["nomic-embed-text", "mxbai-embed-large", "all-minilm"]

# ── Handlers para cada tab ───────────────────────────────────────────────────

def handler_tokens(texto, modelo):
    """Procesa el texto y devuelve info de tokens formateada."""
    resultado = contar_tokens(texto, modelo)

    if "error" in resultado:
        return f"❌ Error: {resultado['error']}"

    return (
        f"🔢 Tokens:          {resultado['tokens']}\n"
        f"📝 Palabras:        {resultado['palabras']}\n"
        f"📊 Tokens/Palabra:  {resultado['ratio_tokens_por_palabra']}\n\n"
        f"💡 Cuanto mayor el ratio, más el modelo 'divide' las palabras en sub-unidades."
    )


def handler_embedding(texto, modelo):
    """Genera el embedding y devuelve preview del vector + datos para el gráfico."""
    if not texto.strip():
        return "⚠️ Ingresa un texto primero.", None

    try:
        vector = obtener_embedding(texto, modelo)
        dim = len(vector)

        # Texto informativo
        preview = vector[:10]  # Primeros 10 valores
        info = (
            f"📐 Dimensiones del vector: {dim}\n"
            f"🔢 Primeros 10 valores:\n"
            + "\n".join([f"  [{i:>3}]: {v:+.6f}" for i, v in enumerate(preview)])
            + f"\n  ... ({dim - 10} valores más)"
        )

        # Datos para el gráfico de barras (primeras 50 dimensiones)
        n_mostrar = min(50, dim)
        datos_grafico = {
            "Dimensión": list(range(n_mostrar)),
            "Valor": vector[:n_mostrar]
        }

        return info, datos_grafico

    except Exception as e:
        return f"❌ Error: {str(e)}", None


def handler_similitud(texto_a, texto_b, modelo):
    """Calcula la similitud coseno entre dos textos."""
    if not texto_a.strip() or not texto_b.strip():
        return "⚠️ Ingresa ambos textos antes de calcular."

    try:
        vec_a = obtener_embedding(texto_a, modelo)
        vec_b = obtener_embedding(texto_b, modelo)
        score = similitud_coseno(vec_a, vec_b)
        interpretacion = interpretar_similitud(score)

        return (
            f"📐 Score de similitud coseno: {score:.4f}\n\n"
            f"{interpretacion}\n\n"
            f"📏 Dimensiones del vector: {len(vec_a)}\n"
            f"🤖 Modelo usado: {modelo}"
        )

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ── Construcción de la interfaz ──────────────────────────────────────────────

with gr.Blocks(
    title="Embeddings con Ollama",
    theme=gr.themes.Soft(),
    css="""
        .output-text { font-family: monospace; font-size: 14px; }
        .gr-tab-label { font-weight: 600; font-size: 15px; }
    """
) as demo:

    gr.Markdown("""
    # 🧠 Explorador de Embeddings con Ollama
    Experimenta con tokens, vectores y similitud semántica en tiempo real.
    """)

    with gr.Tabs():

        # ── Tab 1: Contador de tokens ────────────────────────────────────────
        with gr.Tab("🔢 Tokens"):
            gr.Markdown("""
            ### Cuántos tokens tiene tu texto
            Pega cualquier texto y selecciona el modelo para ver cómo lo tokeniza.
            Prueba el mismo texto con distintos modelos y observa si el conteo varía.
            """)

            with gr.Row():
                with gr.Column(scale=2):
                    txt_tokens = gr.Textbox(
                        label="📝 Tu texto",
                        placeholder="Pega o escribe cualquier texto aquí...",
                        lines=6
                    )
                    modelo_tokens = gr.Dropdown(
                        choices=MODELOS_DISPONIBLES,
                        value="nomic-embed-text",
                        label="🤖 Modelo de embeddings"
                    )
                    btn_tokens = gr.Button("Contar tokens", variant="primary")

                with gr.Column(scale=1):
                    out_tokens = gr.Textbox(
                        label="📊 Resultado",
                        lines=8,
                        interactive=False,
                        elem_classes="output-text"
                    )

            btn_tokens.click(
                fn=handler_tokens,
                inputs=[txt_tokens, modelo_tokens],
                outputs=out_tokens
            )

        # ── Tab 2: Visualizador de embedding ─────────────────────────────────
        with gr.Tab("📊 Embedding"):
            gr.Markdown("""
            ### Visualiza el vector de tu texto
            El modelo convierte tu texto en un vector de cientos de números.
            Aquí verás los primeros 50 valores representados como un gráfico de barras.
            Cada barra es una dimensión del espacio semántico.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    txt_embed = gr.Textbox(
                        label="📝 Tu texto",
                        placeholder="Pega o escribe cualquier texto aquí...",
                        lines=5
                    )
                    modelo_embed = gr.Dropdown(
                        choices=MODELOS_DISPONIBLES,
                        value="nomic-embed-text",
                        label="🤖 Modelo de embeddings"
                    )
                    btn_embed = gr.Button("Generar embedding", variant="primary")
                    out_embed_info = gr.Textbox(
                        label="📐 Info del vector",
                        lines=14,
                        interactive=False,
                        elem_classes="output-text"
                    )

                with gr.Column(scale=2):
                    out_embed_plot = gr.BarPlot(
                        x="Dimensión",
                        y="Valor",
                        title="Primeras 50 dimensiones del vector",
                        label="Vector embedding",
                        color="Valor",
                        height=400
                    )

            btn_embed.click(
                fn=handler_embedding,
                inputs=[txt_embed, modelo_embed],
                outputs=[out_embed_info, out_embed_plot]
            )

        # ── Tab 3: Similitud coseno ───────────────────────────────────────────
        with gr.Tab("🔍 Similitud Coseno"):
            gr.Markdown("""
            ### Compara la similitud entre dos textos
            Ingresa dos textos y el modelo calculará qué tan parecidos son semánticamente.
            El resultado es un número entre -1 (opuestos) y 1 (idénticos).

            **Prueba esto:** compara *"el perro corre"* con *"el can trota"* vs *"la economía global"*.
            """)

            with gr.Row():
                txt_a = gr.Textbox(
                    label="📄 Texto A",
                    placeholder="Primer texto...",
                    lines=5
                )
                txt_b = gr.Textbox(
                    label="📄 Texto B",
                    placeholder="Segundo texto...",
                    lines=5
                )

            with gr.Row():
                modelo_sim = gr.Dropdown(
                    choices=MODELOS_DISPONIBLES,
                    value="nomic-embed-text",
                    label="🤖 Modelo de embeddings",
                    scale=2
                )
                btn_sim = gr.Button(
                    "Calcular similitud",
                    variant="primary",
                    scale=1
                )

            out_sim = gr.Textbox(
                label="📊 Resultado",
                lines=6,
                interactive=False,
                elem_classes="output-text"
            )

            btn_sim.click(
                fn=handler_similitud,
                inputs=[txt_a, txt_b, modelo_sim],
                outputs=out_sim
            )

    gr.Markdown("""
    ---
    > 🛠️ Powered by **Ollama** · **Gradio** · Modelos: nomic-embed-text, mxbai-embed-large, all-minilm
    """)

# Lanzamos la app en modo local (Ngrok la expone en la siguiente celda)
demo.launch(server_name="0.0.0.0", server_port=7860, share=False, quiet=True)
print("✅ Interfaz Gradio corriendo en http://localhost:7860")
print("   → Ejecuta la siguiente celda para obtener la URL pública con Ngrok")

/tmp/ipykernel_1708/3888410057.py:77: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1708/3888410057.py:77: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


<IPython.core.display.Javascript object>

✅ Interfaz Gradio corriendo en http://localhost:7860
   → Ejecuta la siguiente celda para obtener la URL pública con Ngrok


---
## 🌐 Celda 5 — Exponer la interfaz con Ngrok

Colab corre en los servidores de Google, no en tu computadora.  
Para acceder a la interfaz desde cualquier navegador, usamos **Ngrok**: crea un túnel seguro entre internet y el puerto local donde corre Gradio.

### Pasos:
1. Ve a [ngrok.com](https://ngrok.com) → Login → **Your Authtoken**
2. Copia tu token personal
3. Pégalo en la variable `NGROK_TOKEN` abajo
4. Ejecuta la celda → aparecerá tu URL pública

> ⚠️ **Importante:** La URL es temporal y personal. Caduca cuando cierras el notebook.  
> No compartas tu authtoken con nadie: está ligado a tu cuenta.

In [ ]:
from pyngrok import ngrok, conf

# ── 1. Pega aquí tu authtoken de https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "PEGA_TU_TOKEN_AQUÍ"  # <-- reemplaza esto

# ── 2. Validación antes de continuar ────────────────────────────────────────
if NGROK_TOKEN == "PEGA_TU_TOKEN_AQUÍ" or not NGROK_TOKEN.strip():
    print("❌ Debes pegar tu authtoken de Ngrok en la variable NGROK_TOKEN")
    print("   Ve a https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    # ── 3. Configurar y lanzar el túnel ─────────────────────────────────────
    try:
        # Cerramos túneles anteriores si los hay
        ngrok.kill()

        # Autenticamos con el token del estudiante
        conf.get_default().auth_token = NGROK_TOKEN

        # Creamos el túnel apuntando al puerto de Gradio
        tunel = ngrok.connect(7860, "http")
        url_publica = tunel.public_url

        print("="*60)
        print("🌐 TU INTERFAZ ESTÁ DISPONIBLE EN:")
        print(f"\n   👉  {url_publica}\n")
        print("="*60)
        print("✅ Puedes abrir esa URL en cualquier navegador o dispositivo")
        print("⚠️  La URL es temporal: deja de funcionar si cierras el notebook")

    except Exception as e:
        print(f"❌ Error al conectar con Ngrok: {str(e)}")
        print("   Verifica que tu token sea válido y que no tengas otro túnel activo")

---
## 🧪 Actividades de exploración

Una vez que la interfaz esté corriendo, prueba estos experimentos:

### Tab Tokens
- [ ] Pega el mismo párrafo con los 3 modelos. ¿El conteo varía?
- [ ] ¿Cuántos tokens tiene un emoji? ¿Y una palabra técnica como `transformers`?
- [ ] ¿Qué pasa con texto en múltiples idiomas?

### Tab Embedding
- [ ] Genera embeddings de palabras simples: *"gato"*, *"perro"*, *"auto"*
- [ ] ¿El patrón del gráfico cambia mucho entre palabras similares?
- [ ] Compara el vector de `nomic-embed-text` vs `all-minilm` para el mismo texto. ¿Tienen la misma longitud?

### Tab Similitud
- [ ] Prueba: *"el banco del parque"* vs *"el banco donde guardo dinero"*. ¿El modelo distingue el contexto?
- [ ] ¿Qué score obtienes entre *"Hola"* y *"Hello"*?
- [ ] ¿Un texto y su traducción tienen score cercano a 1.0?

---

## 💬 Preguntas de reflexión

1. ¿Por qué crees que el número de tokens no es igual al número de palabras?
2. Si dos palabras tienen un score de similitud de 0.95, ¿significa que son sinónimos perfectos?
3. ¿En qué aplicaciones reales usarías la similitud coseno?
4. ¿Qué ventaja tiene usar un modelo más grande (`mxbai-embed-large`) sobre uno más pequeño (`all-minilm`)?